# Social Listening: Implementation Notebook
## Live Data Collection + NLP Analysis + Network Analysis + Online Reviews

This notebook is designed to run with **free APIs and open datasets** — no paid subscriptions required.

**Sections:**
1. [Environment Setup](#1-environment-setup)
2. [Data Collection — Social Media](#2-data-collection--social-media)
3. [Data Collection — News (GDELT)](#3-data-collection--news-gdelt)
4. [Sentiment Analysis](#4-sentiment-analysis)
5. [Topic Modeling with BERTopic](#5-topic-modeling-with-bertopic)
6. [Named Entity Recognition](#6-named-entity-recognition)
7. [Emotion Analysis](#7-emotion-analysis)
8. [Network & Virality Analysis](#8-network--virality-analysis)
9. [Trend & Anomaly Detection](#9-trend--anomaly-detection)
10. [Online Reviews Analysis](#10-online-reviews-analysis)
11. [Integrated Brand Health Dashboard](#11-integrated-brand-health-dashboard)

---
> **API Keys needed**: Reddit (free PRAW), YouTube (free), optionally HuggingFace token  
> **No key needed**: HackerNews, GDELT, Mastodon, Amazon Reviews dataset, Yelp dataset

---
## 1. Environment Setup

In [ ]:
# Install all required packages
# Run this cell first — it may take a few minutes

!pip install -q praw requests gdeltdoc vaderSentiment transformers \
    bertopic sentence-transformers umap-learn hdbscan \
    spacy nrclex networkx python-louvain ndlib \
    prophet adtk scikit-learn datasets \
    plotly wordcloud pandas numpy tqdm python-dotenv

# Download spaCy English model
!python -m spacy download en_core_web_sm -q

print("Setup complete!")

In [ ]:
import os
import json
import time
import warnings
from datetime import datetime, timedelta
from collections import Counter

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
from tqdm import tqdm
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)

# ── LOAD CREDENTIALS FROM .env ─────────────────────────────────────────────────
# Create a .env file in this directory (see .env.example) — never commit it to git
load_dotenv()

REDDIT_CLIENT_ID     = os.getenv("REDDIT_CLIENT_ID", "")
REDDIT_CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET", "")
REDDIT_USER_AGENT    = os.getenv("REDDIT_USER_AGENT", "SocialListeningBot/1.0")
YOUTUBE_API_KEY      = os.getenv("YOUTUBE_API_KEY", "")

# Validate that credentials are present
missing = [name for name, val in {
    "REDDIT_CLIENT_ID": REDDIT_CLIENT_ID,
    "REDDIT_CLIENT_SECRET": REDDIT_CLIENT_SECRET,
}.items() if not val]

if missing:
    print(f"Warning: missing credentials: {missing}")
    print("Create a .env file with these variables (see .env.example).")
else:
    print("Credentials loaded successfully.")

# ── TOPIC CONFIGURATION ────────────────────────────────────────────────────────
TOPIC       = "artificial intelligence"   # Change to your brand/topic
TOPIC_SHORT = "AI"

print(f"Tracking topic: '{TOPIC}'")
print(f"Date: {datetime.now().strftime('%Y-%m-%d')}")

---
## 2. Data Collection — Social Media

### 2.1 Reddit (via PRAW)

**Setup:** Create a free app at https://www.reddit.com/prefs/apps  
1. Click "create an app"
2. Name: anything, Type: script
3. Redirect URI: http://localhost:8080
4. Copy `client_id` (under app name) and `secret`

In [ ]:
import praw

def get_reddit_posts(subreddit_name: str, query: str, limit: int = 100) -> pd.DataFrame:
    """
    Collect posts from a subreddit matching a search query.
    Free tier: 100 req/min, max 1000 posts per listing.
    """
    reddit = praw.Reddit(
        client_id=REDDIT_CLIENT_ID,
        client_secret=REDDIT_CLIENT_SECRET,
        user_agent=REDDIT_USER_AGENT
    )
    
    records = []
    subreddit = reddit.subreddit(subreddit_name)
    
    for post in subreddit.search(query, limit=limit, sort="new"):
        records.append({
            "id": post.id,
            "source": "reddit",
            "subreddit": subreddit_name,
            "title": post.title,
            "text": post.selftext,
            "full_text": f"{post.title}. {post.selftext}".strip(),
            "score": post.score,
            "upvote_ratio": post.upvote_ratio,
            "num_comments": post.num_comments,
            "created_utc": datetime.fromtimestamp(post.created_utc),
            "url": f"https://reddit.com{post.permalink}",
            "author": str(post.author) if post.author else "[deleted]",
        })
    
    return pd.DataFrame(records)


def get_reddit_comments(subreddit_name: str, query: str, limit: int = 50) -> pd.DataFrame:
    """
    Collect comments from the top posts matching a query.
    """
    reddit = praw.Reddit(
        client_id=REDDIT_CLIENT_ID,
        client_secret=REDDIT_CLIENT_SECRET,
        user_agent=REDDIT_USER_AGENT
    )
    
    records = []
    subreddit = reddit.subreddit(subreddit_name)
    
    for post in subreddit.search(query, limit=10, sort="hot"):
        post.comments.replace_more(limit=0)  # Flatten comment tree
        for comment in post.comments.list()[:limit // 10]:
            if len(comment.body) > 20:  # Filter out very short comments
                records.append({
                    "id": comment.id,
                    "source": "reddit_comment",
                    "parent_post_id": post.id,
                    "subreddit": subreddit_name,
                    "full_text": comment.body,
                    "score": comment.score,
                    "created_utc": datetime.fromtimestamp(comment.created_utc),
                    "author": str(comment.author) if comment.author else "[deleted]",
                })
    
    return pd.DataFrame(records)


# Collect from multiple subreddits for broader coverage
subreddits = ["technology", "MachineLearning", "artificial", "ChatGPT"]

reddit_dfs = []
for sub in subreddits:
    try:
        df = get_reddit_posts(sub, TOPIC, limit=50)
        print(f"r/{sub}: {len(df)} posts")
        reddit_dfs.append(df)
        time.sleep(1)  # Be polite to the API
    except Exception as e:
        print(f"r/{sub}: Error — {e}")

df_reddit = pd.concat(reddit_dfs, ignore_index=True).drop_duplicates(subset="id")
print(f"\nTotal Reddit posts: {len(df_reddit)}")
df_reddit.head(3)

### 2.2 HackerNews (No API Key Required)

In [ ]:
def get_hackernews_posts(query: str, limit: int = 100) -> pd.DataFrame:
    """
    Search HackerNews via Algolia API — completely free, no authentication.
    Returns stories, Ask HN posts, and comments matching the query.
    """
    url = "https://hn.algolia.com/api/v1/search"
    params = {
        "query": query,
        "tags": "story",          # Filter to stories only
        "hitsPerPage": limit,
        "attributesToRetrieve": "objectID,title,url,points,num_comments,created_at,author"
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    records = []
    for hit in data.get("hits", []):
        records.append({
            "id": hit.get("objectID"),
            "source": "hackernews",
            "full_text": hit.get("title", ""),
            "score": hit.get("points", 0),
            "num_comments": hit.get("num_comments", 0),
            "created_utc": pd.to_datetime(hit.get("created_at")),
            "url": hit.get("url", f"https://news.ycombinator.com/item?id={hit.get('objectID')}"),
            "author": hit.get("author", ""),
        })
    
    return pd.DataFrame(records)


def get_hackernews_comments(query: str, limit: int = 100) -> pd.DataFrame:
    """Search HackerNews comments for deeper opinion data."""
    url = "https://hn.algolia.com/api/v1/search"
    params = {
        "query": query,
        "tags": "comment",
        "hitsPerPage": limit,
    }
    response = requests.get(url, params=params)
    data = response.json()
    
    records = []
    for hit in data.get("hits", []):
        comment_text = hit.get("comment_text", "") or ""
        # Strip HTML tags
        import re
        clean_text = re.sub(r"<[^>]+>", " ", comment_text).strip()
        if len(clean_text) > 30:
            records.append({
                "id": hit.get("objectID"),
                "source": "hackernews_comment",
                "full_text": clean_text,
                "score": hit.get("points", 0),
                "created_utc": pd.to_datetime(hit.get("created_at")),
                "author": hit.get("author", ""),
            })
    
    return pd.DataFrame(records)


df_hn = get_hackernews_posts(TOPIC, limit=100)
df_hn_comments = get_hackernews_comments(TOPIC, limit=100)

print(f"HackerNews stories: {len(df_hn)}")
print(f"HackerNews comments: {len(df_hn_comments)}")
df_hn.sort_values("score", ascending=False).head(5)

### 2.3 Mastodon (No API Key Required for Public Timeline)

In [ ]:
def get_mastodon_posts(query: str, limit: int = 40, instance: str = "mastodon.social") -> pd.DataFrame:
    """
    Search Mastodon public posts. No authentication needed for public searches.
    """
    url = f"https://{instance}/api/v2/search"
    params = {"q": query, "type": "statuses", "limit": limit}
    headers = {"User-Agent": "SocialListeningBot/1.0"}
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Mastodon error: {e}")
        return pd.DataFrame()
    
    records = []
    import re
    for status in data.get("statuses", []):
        # Clean HTML from toot content
        content = re.sub(r"<[^>]+>", " ", status.get("content", "")).strip()
        if len(content) > 10:
            records.append({
                "id": status.get("id"),
                "source": "mastodon",
                "full_text": content,
                "score": status.get("favourites_count", 0),
                "reblogs": status.get("reblogs_count", 0),
                "replies": status.get("replies_count", 0),
                "created_utc": pd.to_datetime(status.get("created_at")),
                "author": status.get("account", {}).get("acct", ""),
                "url": status.get("url", ""),
            })
    
    return pd.DataFrame(records)


df_mastodon = get_mastodon_posts(TOPIC, limit=40)
print(f"Mastodon toots: {len(df_mastodon)}")
if not df_mastodon.empty:
    df_mastodon.head(3)

In [ ]:
# Combine all social media data into a single DataFrame
social_sources = [df_hn, df_hn_comments, df_mastodon]
if not df_reddit.empty:
    social_sources.append(df_reddit)

df_social = pd.concat(
    [df for df in social_sources if not df.empty],
    ignore_index=True
)

# Ensure we have the core columns
df_social["full_text"] = df_social["full_text"].fillna("").astype(str)
df_social["score"] = df_social["score"].fillna(0).astype(int)

# Filter out very short texts
df_social = df_social[df_social["full_text"].str.len() > 20].reset_index(drop=True)

print(f"Total combined social posts: {len(df_social)}")
print("\nBreakdown by source:")
print(df_social["source"].value_counts())

# Quick preview
df_social[["source", "full_text", "score"]].head(5)

---
## 3. Data Collection — News (GDELT)

GDELT is a **completely free** global news database monitoring 100+ languages in near real-time.

In [ ]:
from gdeltdoc import GdeltDoc, Filters

def get_gdelt_news(query: str, days_back: int = 30) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Search GDELT for news articles and get volume timeline.
    Free, no API key. Rolling 3-month window.
    Returns: (articles_df, timeline_df)
    """
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days_back)
    
    f = Filters(
        keyword=query,
        start_date=start_date.strftime("%Y-%m-%d"),
        end_date=end_date.strftime("%Y-%m-%d"),
        num_records=250,
    )
    
    gd = GdeltDoc()
    
    try:
        articles = gd.article_search(f)
        timeline = gd.timeline_search("timelinevol", f)  # Volume over time
    except Exception as e:
        print(f"GDELT error: {e}")
        return pd.DataFrame(), pd.DataFrame()
    
    if not articles.empty:
        articles["source"] = "gdelt_news"
        articles["full_text"] = articles.get("title", pd.Series(dtype=str))
        articles["created_utc"] = pd.to_datetime(articles.get("seendate", pd.Series(dtype=str)))
    
    return articles, timeline


print(f"Fetching GDELT news for '{TOPIC}'...")
df_gdelt, df_gdelt_timeline = get_gdelt_news(TOPIC, days_back=30)

print(f"GDELT articles found: {len(df_gdelt)}")
if not df_gdelt_timeline.empty:
    print(f"Timeline data points: {len(df_gdelt_timeline)}")
    
if not df_gdelt.empty:
    df_gdelt[["title", "url", "sourcecountry", "created_utc"]].head(5)

In [ ]:
# Plot GDELT news volume over time
if not df_gdelt_timeline.empty:
    fig = px.line(
        df_gdelt_timeline,
        x="datetime",
        y="Volume Intensity",
        title=f"News Volume for '{TOPIC}' — Last 30 Days (GDELT)",
        labels={"datetime": "Date", "Volume Intensity": "Article Volume"},
        template="plotly_white"
    )
    fig.update_traces(line_color="#1f77b4", line_width=2)
    fig.show()
else:
    print("No timeline data to plot. Try a more common keyword.")

In [ ]:
# Geographic distribution of news coverage
if not df_gdelt.empty and "sourcecountry" in df_gdelt.columns:
    country_counts = df_gdelt["sourcecountry"].value_counts().head(15).reset_index()
    country_counts.columns = ["Country", "Article Count"]
    
    fig = px.bar(
        country_counts,
        x="Article Count", y="Country",
        orientation="h",
        title=f"News Coverage by Country — '{TOPIC}'",
        template="plotly_white",
        color="Article Count",
        color_continuous_scale="Blues"
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()

---
## 4. Sentiment Analysis

We use VADER (fast, rule-based) and optionally Twitter-RoBERTa (transformer-based) for comparison.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize VADER
sia = SentimentIntensityAnalyzer()

def analyze_sentiment_vader(texts: list[str]) -> pd.DataFrame:
    """
    Apply VADER sentiment to a list of texts.
    Returns DataFrame with compound, positive, negative, neutral scores
    and a sentiment label.
    """
    results = []
    for text in texts:
        scores = sia.polarity_scores(str(text))
        compound = scores["compound"]
        # Standard VADER thresholds
        if compound >= 0.05:
            label = "positive"
        elif compound <= -0.05:
            label = "negative"
        else:
            label = "neutral"
        
        results.append({
            "compound": compound,
            "positive": scores["pos"],
            "negative": scores["neg"],
            "neutral": scores["neu"],
            "sentiment": label,
        })
    return pd.DataFrame(results)


# Apply to social data
print("Running VADER sentiment analysis...")
sentiment_df = analyze_sentiment_vader(df_social["full_text"].tolist())
df_social = pd.concat([df_social.reset_index(drop=True), sentiment_df], axis=1)

# Summary
sentiment_counts = df_social["sentiment"].value_counts()
total = len(df_social)

print("\nSentiment Distribution:")
for sentiment, count in sentiment_counts.items():
    pct = count / total * 100
    print(f"  {sentiment:10s}: {count:4d} ({pct:.1f}%)")

# Net Sentiment Score
pos = sentiment_counts.get("positive", 0)
neg = sentiment_counts.get("negative", 0)
nss = ((pos - neg) / total) * 100
print(f"\nNet Sentiment Score (NSS): {nss:.1f}  (range: -100 to +100)")

In [ ]:
# Sentiment distribution plot
fig = px.pie(
    values=sentiment_counts.values,
    names=sentiment_counts.index,
    title=f"Sentiment Distribution — '{TOPIC}' (n={total})",
    color=sentiment_counts.index,
    color_discrete_map={"positive": "#2ecc71", "neutral": "#95a5a6", "negative": "#e74c3c"},
    hole=0.4,
    template="plotly_white"
)
fig.show()

In [ ]:
# Sentiment over time
if "created_utc" in df_social.columns:
    df_time = df_social.copy()
    df_time["created_utc"] = pd.to_datetime(df_time["created_utc"], utc=True, errors="coerce")
    df_time = df_time.dropna(subset=["created_utc"])
    df_time["date"] = df_time["created_utc"].dt.date
    
    daily_sentiment = (
        df_time.groupby(["date", "sentiment"])
        .size()
        .reset_index(name="count")
    )
    
    fig = px.bar(
        daily_sentiment,
        x="date", y="count", color="sentiment",
        title=f"Daily Sentiment Volume — '{TOPIC}'",
        color_discrete_map={"positive": "#2ecc71", "neutral": "#95a5a6", "negative": "#e74c3c"},
        template="plotly_white",
        barmode="stack"
    )
    fig.show()

In [ ]:
# Optional: Twitter-RoBERTa for higher accuracy
# Uncomment to use — requires ~500MB model download

# from transformers import pipeline
#
# roberta_sentiment = pipeline(
#     "sentiment-analysis",
#     model="cardiffnlp/twitter-roberta-base-sentiment-latest",
#     max_length=512, truncation=True
# )
#
# # Run on a sample (transformer inference is slower)
# sample_texts = df_social["full_text"].head(50).tolist()
# roberta_results = roberta_sentiment(sample_texts, batch_size=8)
# df_roberta_sample = pd.DataFrame(roberta_results)
# df_roberta_sample["text"] = sample_texts
# print(df_roberta_sample["label"].value_counts())

---
## 5. Topic Modeling with BERTopic

BERTopic uses sentence embeddings + UMAP + HDBSCAN — ideal for short social media texts.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Prepare texts — filter to meaningful length
docs = df_social[df_social["full_text"].str.len() > 40]["full_text"].tolist()

if len(docs) < 10:
    print("Not enough documents for topic modeling (need 10+). Collect more data first.")
else:
    print(f"Running BERTopic on {len(docs)} documents...")
    
    # Use a lightweight embedding model for speed
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    
    topic_model = BERTopic(
        embedding_model=embedding_model,
        nr_topics="auto",        # Automatically determine number of topics
        min_topic_size=3,        # Minimum docs per topic
        language="english",
        calculate_probabilities=False,
        verbose=False
    )
    
    topics, _ = topic_model.fit_transform(docs)
    
    topic_info = topic_model.get_topic_info()
    # -1 is the outlier topic — exclude it
    topic_info_filtered = topic_info[topic_info["Topic"] != -1]
    
    print(f"\nDiscovered {len(topic_info_filtered)} topics:")
    print(topic_info_filtered[["Topic", "Count", "Name"]].to_string(index=False))

In [ ]:
# Visualize topic hierarchy
if len(docs) >= 10:
    # Top keywords per topic
    for topic_id in topic_info_filtered["Topic"].head(8).tolist():
        keywords = topic_model.get_topic(topic_id)
        top_words = ", ".join([word for word, _ in keywords[:6]])
        count = topic_info_filtered.loc[
            topic_info_filtered["Topic"] == topic_id, "Count"
        ].values[0]
        print(f"Topic {topic_id:3d} ({count:4d} docs): {top_words}")

In [ ]:
# Interactive 2D topic map
if len(docs) >= 10 and len(topic_info_filtered) >= 2:
    fig = topic_model.visualize_topics()
    fig.update_layout(title=f"Topic Map — '{TOPIC}' discussions")
    fig.show()

In [ ]:
# Word scores barchart per topic
if len(docs) >= 10 and len(topic_info_filtered) >= 2:
    fig = topic_model.visualize_barchart(top_n_topics=6)
    fig.show()

In [ ]:
# Attach topics back to posts
if len(docs) >= 10:
    df_with_topics = df_social[df_social["full_text"].str.len() > 40].copy().reset_index(drop=True)
    df_with_topics["topic_id"] = topics
    
    # Add topic name
    topic_name_map = dict(zip(topic_info["Topic"], topic_info["Name"]))
    df_with_topics["topic_name"] = df_with_topics["topic_id"].map(topic_name_map)
    
    # Sentiment per topic
    topic_sentiment = (
        df_with_topics[df_with_topics["topic_id"] != -1]
        .groupby(["topic_name", "sentiment"])
        .size()
        .reset_index(name="count")
    )
    
    fig = px.bar(
        topic_sentiment,
        x="topic_name", y="count", color="sentiment",
        title="Sentiment Distribution by Topic",
        color_discrete_map={"positive": "#2ecc71", "neutral": "#95a5a6", "negative": "#e74c3c"},
        template="plotly_white",
        barmode="group"
    )
    fig.update_xaxes(tickangle=45)
    fig.show()

---
## 6. Named Entity Recognition

Identify brands, people, organizations, and locations mentioned alongside your topic.

In [ ]:
import spacy
from collections import defaultdict
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Optional: add custom brand/entity patterns
# ruler = nlp.add_pipe("entity_ruler", before="ner")
# custom_patterns = [
#     {"label": "BRAND", "pattern": "your_brand_name"},
#     {"label": "PRODUCT", "pattern": "your_product_name"},
# ]
# ruler.add_patterns(custom_patterns)


def extract_entities(texts: list[str], batch_size: int = 50) -> pd.DataFrame:
    """
    Extract named entities from a list of texts using spaCy.
    Returns DataFrame with entity text, label, and source text index.
    """
    entities = defaultdict(list)
    
    for i, doc in enumerate(nlp.pipe(texts, batch_size=batch_size)):
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT", "EVENT", "WORK_OF_ART"]:
                entities["text_idx"].append(i)
                entities["entity"].append(ent.text)
                entities["label"].append(ent.label_)
    
    return pd.DataFrame(entities)


# Run on a sample for speed (increase for full analysis)
sample_texts = df_social["full_text"].head(200).tolist()
print(f"Extracting entities from {len(sample_texts)} texts...")

df_entities = extract_entities(sample_texts)

print(f"\nTotal entity mentions: {len(df_entities)}")
print("\nTop 15 entities overall:")
print(df_entities["entity"].value_counts().head(15).to_string())

In [ ]:
# Entity breakdown by type
for label in ["ORG", "PERSON", "GPE", "PRODUCT"]:
    subset = df_entities[df_entities["label"] == label]
    if not subset.empty:
        top = subset["entity"].value_counts().head(8)
        print(f"\nTop {label}:")
        for entity, count in top.items():
            print(f"  {entity:30s} {count:4d}")

In [ ]:
# Word cloud of most mentioned organizations
org_counts = df_entities[df_entities["label"] == "ORG"]["entity"].value_counts()

if len(org_counts) > 3:
    wc = WordCloud(
        width=800, height=400,
        background_color="white",
        colormap="Blues",
        max_words=50
    ).generate_from_frequencies(org_counts.to_dict())
    
    plt.figure(figsize=(12, 5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Most Mentioned Organizations in '{TOPIC}' Discussions", fontsize=14)
    plt.tight_layout()
    plt.show()

---
## 7. Emotion Analysis

Go beyond positive/negative — detect joy, anger, fear, sadness, surprise, trust, disgust, anticipation.

In [ ]:
from nrclex import NRCLex

def analyze_emotions_nrc(texts: list[str]) -> pd.DataFrame:
    """
    Apply NRC Emotion Lexicon to texts.
    Fast, rule-based, covers 8 Plutchik emotions + positive/negative.
    """
    emotion_cols = ["fear", "anger", "anticipation", "trust", 
                    "surprise", "positive", "negative", "sadness", 
                    "disgust", "joy"]
    
    results = []
    for text in texts:
        try:
            emotion = NRCLex(str(text))
            freqs = emotion.affect_frequencies
            row = {col: freqs.get(col, 0.0) for col in emotion_cols}
            # Dominant emotion
            emotion_only = {k: v for k, v in row.items() 
                           if k not in ["positive", "negative"]}
            row["dominant_emotion"] = max(emotion_only, key=emotion_only.get) \
                                      if any(v > 0 for v in emotion_only.values()) else "neutral"
        except Exception:
            row = {col: 0.0 for col in emotion_cols}
            row["dominant_emotion"] = "neutral"
        results.append(row)
    
    return pd.DataFrame(results)


print("Running NRC emotion analysis...")
emotion_df = analyze_emotions_nrc(df_social["full_text"].head(500).tolist())
df_emotion_sample = df_social.head(500).copy().reset_index(drop=True)
df_emotion_sample = pd.concat([df_emotion_sample, emotion_df], axis=1)

print("\nDominant emotion distribution:")
print(df_emotion_sample["dominant_emotion"].value_counts())

In [ ]:
# Emotion radar chart — average emotion scores
emotion_cols = ["fear", "anger", "anticipation", "trust", 
                "surprise", "sadness", "disgust", "joy"]

avg_emotions = emotion_df[emotion_cols].mean()

fig = go.Figure(data=go.Scatterpolar(
    r=avg_emotions.values,
    theta=avg_emotions.index,
    fill="toself",
    name="Emotion Profile",
    line_color="#3498db",
    fillcolor="rgba(52, 152, 219, 0.3)"
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    showlegend=False,
    title=f"Emotion Profile — '{TOPIC}' Discussions",
    template="plotly_white"
)
fig.show()

In [ ]:
# Optional: GoEmotions (27 fine-grained emotions) — more accurate
# Uncomment to use (requires ~400MB model download)

# from transformers import pipeline
#
# go_emotions = pipeline(
#     "text-classification",
#     model="bhadresh-savani/bert-base-uncased-emotion",
#     return_all_scores=True, top_k=None
# )
#
# sample_5 = df_social["full_text"].head(5).tolist()
# results = go_emotions(sample_5, truncation=True, max_length=512)
#
# for text, result in zip(sample_5, results):
#     top_emotions = sorted(result, key=lambda x: x["score"], reverse=True)[:3]
#     print(f"Text: {text[:60]}...")
#     for e in top_emotions:
#         print(f"  {e['label']:20s}: {e['score']:.3f}")
#     print()

---
## 8. Network & Virality Analysis

Build a co-mention network and analyze information diffusion patterns.

In [ ]:
import networkx as nx

# ── 8.1 Engagement / Virality Scoring ──────────────────────────────────────────

def compute_virality_score(row: pd.Series) -> float:
    """
    Composite virality score using engagement signals.
    Higher score = higher potential to spread.
    """
    score = row.get("score", 0) or 0
    comments = row.get("num_comments", 0) or 0
    reblogs = row.get("reblogs", 0) or 0
    replies = row.get("replies", 0) or 0
    
    # Weighted composite: shares/reblogs are strongest signal
    return score * 0.3 + comments * 0.4 + (reblogs + replies) * 0.3


df_social["virality_score"] = df_social.apply(compute_virality_score, axis=1)

# Top viral posts
print("Top 10 most viral posts:")
df_social.sort_values("virality_score", ascending=False)[
    ["source", "full_text", "score", "virality_score", "sentiment"]
].head(10).assign(
    full_text=lambda x: x["full_text"].str[:80] + "..."
).to_string(index=False)

In [ ]:
print(df_social.sort_values("virality_score", ascending=False)[
    ["source", "full_text", "score", "virality_score", "sentiment"]
].head(10).assign(
    full_text=lambda x: x["full_text"].str[:80] + "..."
).to_string(index=False))

In [ ]:
# ── 8.2 Author Co-mention Network ──────────────────────────────────────────────
# Build a network where nodes = authors, edges = posting about same entity

def build_author_entity_network(df: pd.DataFrame, entity_df: pd.DataFrame) -> nx.Graph:
    """
    Build a bipartite graph: authors ↔ entities they mentioned.
    Then project onto author-author network (connected if they mention same entities).
    """
    G = nx.Graph()
    
    # Add author nodes
    authors = df["author"].dropna().unique()
    for author in authors:
        G.add_node(author, node_type="author")
    
    # Find which authors mentioned which entities
    entity_counts = entity_df["entity"].value_counts()
    top_entities = entity_counts[entity_counts >= 2].index.tolist()[:30]
    
    for entity in top_entities:
        # Find text indices where this entity appears
        text_indices = entity_df[entity_df["entity"] == entity]["text_idx"].tolist()
        # Get authors for those text indices
        entity_authors = []
        for idx in text_indices:
            if idx < len(df) and pd.notna(df.iloc[idx].get("author", None)):
                author = df.iloc[idx]["author"]
                if author and author != "[deleted]":
                    entity_authors.append(author)
        
        # Create edges between authors who mentioned the same entity
        for i in range(len(entity_authors)):
            for j in range(i + 1, len(entity_authors)):
                if entity_authors[i] != entity_authors[j]:
                    if G.has_edge(entity_authors[i], entity_authors[j]):
                        G[entity_authors[i]][entity_authors[j]]["weight"] += 1
                    else:
                        G.add_edge(entity_authors[i], entity_authors[j], weight=1)
    
    return G


# Build network (using the subset where we have entity data)
df_social_sample = df_social.head(200).reset_index(drop=True)
G_authors = build_author_entity_network(df_social_sample, df_entities)

print(f"Author network: {G_authors.number_of_nodes()} nodes, {G_authors.number_of_edges()} edges")

In [ ]:
# ── 8.3 Network Centrality Analysis ───────────────────────────────────────────

if G_authors.number_of_nodes() > 3 and G_authors.number_of_edges() > 0:
    # Compute centrality metrics
    pagerank = nx.pagerank(G_authors, weight="weight")
    betweenness = nx.betweenness_centrality(G_authors, weight="weight", normalized=True)
    degree = nx.degree_centrality(G_authors)
    
    centrality_df = pd.DataFrame({
        "author": list(pagerank.keys()),
        "pagerank": list(pagerank.values()),
        "betweenness": [betweenness.get(n, 0) for n in pagerank.keys()],
        "degree": [degree.get(n, 0) for n in pagerank.keys()],
    }).sort_values("pagerank", ascending=False)
    
    print("Top influencers by PageRank (quality of connections):")
    print(centrality_df.head(10).to_string(index=False))
    
    print("\nTop bridge nodes by Betweenness (information brokers):")
    print(centrality_df.sort_values("betweenness", ascending=False).head(10).to_string(index=False))
else:
    print("Network too sparse for meaningful centrality analysis.")
    print("Collect more data or expand entity extraction.")

In [ ]:
# ── 8.4 Community Detection ────────────────────────────────────────────────────

try:
    import community as community_louvain
except ImportError:
    !pip install -q python-louvain
    import community as community_louvain


if G_authors.number_of_edges() > 3:
    partition = community_louvain.best_partition(G_authors)
    community_counts = Counter(partition.values())
    
    print(f"\nDetected {len(community_counts)} communities:")
    for comm_id, size in sorted(community_counts.items(), key=lambda x: -x[1]):
        members = [node for node, c in partition.items() if c == comm_id]
        print(f"  Community {comm_id}: {size} members — {', '.join(members[:5])}{'...' if size > 5 else ''}")
    
    # Modularity score (higher = better-defined communities)
    modularity = community_louvain.modularity(partition, G_authors)
    print(f"\nModularity score: {modularity:.3f}  (0 = random, 1 = perfect communities)")
else:
    print("Not enough edges for community detection.")

In [ ]:
# ── 8.5 Information Diffusion Simulation ────────────────────────────────────────
# Simulate how content spreads using an SIR model

try:
    import ndlib.models.ModelConfig as mc
    import ndlib.models.epidemics as ep
    HAS_NDLIB = True
except ImportError:
    !pip install -q ndlib
    try:
        import ndlib.models.ModelConfig as mc
        import ndlib.models.epidemics as ep
        HAS_NDLIB = True
    except ImportError:
        HAS_NDLIB = False
        print("NDlib not available — skipping diffusion simulation")


if HAS_NDLIB and G_authors.number_of_nodes() > 5:
    # SIR model: Susceptible → Infected (shares content) → Recovered (stops sharing)
    model = ep.SIRModel(G_authors)
    
    config = mc.Configuration()
    config.add_model_parameter("beta", 0.3)   # Infection rate (sharing probability)
    config.add_model_parameter("gamma", 0.1)  # Recovery rate (stops sharing)
    config.add_model_parameter("fraction_infected", 0.05)  # 5% initially share
    
    model.set_initial_status(config)
    
    # Run simulation
    iterations = model.iteration_bunch(50)
    
    # Parse results
    sim_data = []
    for i, it in enumerate(iterations):
        node_count = it.get("node_count", {})
        sim_data.append({
            "step": i,
            "susceptible": node_count.get(0, 0),
            "infected": node_count.get(1, 0),
            "recovered": node_count.get(2, 0),
        })
    
    df_sim = pd.DataFrame(sim_data)
    
    fig = px.line(
        df_sim, x="step",
        y=["susceptible", "infected", "recovered"],
        title="SIR Information Diffusion Simulation",
        labels={"step": "Time Step", "value": "Number of Nodes", "variable": "State"},
        color_discrete_map={
            "susceptible": "#3498db",
            "infected": "#e74c3c",
            "recovered": "#2ecc71"
        },
        template="plotly_white"
    )
    fig.show()
    
    peak_infected = df_sim["infected"].max()
    peak_step = df_sim["infected"].idxmax()
    print(f"\nPeak spread: {peak_infected} nodes at step {peak_step}")
    print(f"Final recovered (total reached): {df_sim['recovered'].iloc[-1]}")

---
## 9. Trend & Anomaly Detection

In [ ]:
# ── 9.1 Mention Volume Time Series ────────────────────────────────────────────

df_ts = df_social.copy()
df_ts["created_utc"] = pd.to_datetime(df_ts["created_utc"], utc=True, errors="coerce")
df_ts = df_ts.dropna(subset=["created_utc"])
df_ts["date"] = df_ts["created_utc"].dt.date

# Daily mention counts
daily_counts = df_ts.groupby("date").size().reset_index(name="count")
daily_counts["date"] = pd.to_datetime(daily_counts["date"])
daily_counts = daily_counts.sort_values("date")

# Daily average sentiment
daily_sentiment_avg = df_ts.groupby("date")["compound"].mean().reset_index()
daily_sentiment_avg["date"] = pd.to_datetime(daily_sentiment_avg["date"])

print(f"Time series spans: {daily_counts['date'].min()} to {daily_counts['date'].max()}")
print(f"Total days: {len(daily_counts)}")
daily_counts.tail(7)

In [ ]:
# ── 9.2 Prophet Forecasting ────────────────────────────────────────────────────

try:
    from prophet import Prophet
    HAS_PROPHET = True
except ImportError:
    !pip install -q prophet
    from prophet import Prophet
    HAS_PROPHET = True


if len(daily_counts) >= 5:
    # Prophet requires columns 'ds' and 'y'
    prophet_df = daily_counts.rename(columns={"date": "ds", "count": "y"})
    
    m = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=True,
        daily_seasonality=False,
        interval_width=0.95
    )
    m.fit(prophet_df)
    
    # Forecast 14 days ahead
    future = m.make_future_dataframe(periods=14)
    forecast = m.predict(future)
    
    # Plot using Plotly
    fig = go.Figure()
    
    # Historical
    fig.add_trace(go.Scatter(
        x=prophet_df["ds"], y=prophet_df["y"],
        mode="markers", name="Actual",
        marker=dict(color="#2c3e50", size=6)
    ))
    
    # Forecast
    fig.add_trace(go.Scatter(
        x=forecast["ds"], y=forecast["yhat"],
        mode="lines", name="Forecast",
        line=dict(color="#3498db", width=2)
    ))
    
    # Confidence interval
    fig.add_trace(go.Scatter(
        x=list(forecast["ds"]) + list(forecast["ds"])[::-1],
        y=list(forecast["yhat_upper"]) + list(forecast["yhat_lower"])[::-1],
        fill="toself",
        fillcolor="rgba(52, 152, 219, 0.15)",
        line=dict(color="rgba(255,255,255,0)"),
        name="95% CI"
    ))
    
    fig.update_layout(
        title=f"Mention Volume Forecast — '{TOPIC}' (+14 days)",
        xaxis_title="Date",
        yaxis_title="Daily Mentions",
        template="plotly_white"
    )
    fig.show()
else:
    print("Not enough time series data for forecasting (need 5+ days). Collect more data.")

In [ ]:
# ── 9.3 Anomaly Detection ─────────────────────────────────────────────────────
# Detect unusual spikes in mention volume using Z-score method

def detect_spikes(series: pd.Series, window: int = 7, threshold: float = 2.5) -> pd.Series:
    """
    Detect anomalous spikes using rolling Z-score.
    Returns boolean Series: True = anomaly detected.
    """
    rolling_mean = series.rolling(window=window, min_periods=2, center=True).mean()
    rolling_std = series.rolling(window=window, min_periods=2, center=True).std()
    z_scores = (series - rolling_mean) / (rolling_std + 1e-8)  # avoid division by zero
    return z_scores.abs() > threshold


if len(daily_counts) >= 5:
    daily_counts_indexed = daily_counts.set_index("date")["count"]
    anomalies = detect_spikes(daily_counts_indexed, window=5, threshold=2.0)
    
    spike_days = daily_counts[anomalies.values]
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=daily_counts["date"], y=daily_counts["count"],
        mode="lines+markers", name="Daily Mentions",
        line=dict(color="#3498db", width=2)
    ))
    
    if not spike_days.empty:
        fig.add_trace(go.Scatter(
            x=spike_days["date"], y=spike_days["count"],
            mode="markers", name="Anomaly / Spike",
            marker=dict(color="#e74c3c", size=12, symbol="star")
        ))
        print(f"\nDetected {len(spike_days)} anomalous days:")
        print(spike_days.to_string(index=False))
    else:
        print("No anomalies detected in the current dataset.")
    
    fig.update_layout(
        title=f"Mention Volume with Anomaly Detection — '{TOPIC}'",
        xaxis_title="Date",
        yaxis_title="Daily Mentions",
        template="plotly_white"
    )
    fig.show()

---
## 10. Online Reviews Analysis

Working with two free datasets: **Amazon Reviews 2023** and **Yelp Open Dataset**.

In [ ]:
# ── 10.1 Amazon Reviews Dataset ───────────────────────────────────────────────
# Streaming directly from HuggingFace — no download needed
# Categories: Electronics, Books, Clothing_Shoes_and_Jewelry, Home_and_Kitchen,
#             Sports_and_Outdoors, Beauty_and_Personal_Care, etc.

from datasets import load_dataset

REVIEW_CATEGORY = "raw_review_Electronics"  # Change to any category
REVIEW_SAMPLE_SIZE = 2000

print(f"Loading Amazon Reviews 2023 — {REVIEW_CATEGORY}")
print("(Streaming first 2,000 reviews — adjust REVIEW_SAMPLE_SIZE as needed)\n")

try:
    dataset = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        REVIEW_CATEGORY,
        trust_remote_code=True,
        streaming=True  # Streaming mode avoids full download
    )
    
    reviews = []
    for i, row in enumerate(dataset["full"]):
        if i >= REVIEW_SAMPLE_SIZE:
            break
        reviews.append({
            "review_id": row.get("parent_asin", "") + str(i),
            "source": "amazon",
            "rating": row.get("rating", 3),
            "title": row.get("title", ""),
            "full_text": f"{row.get('title', '')} {row.get('text', '')}".strip(),
            "text": row.get("text", ""),
            "helpful_vote": row.get("helpful_vote", 0),
            "verified_purchase": row.get("verified_purchase", False),
            "timestamp": row.get("timestamp", 0),
            "asin": row.get("parent_asin", ""),
        })
    
    df_amazon = pd.DataFrame(reviews)
    df_amazon["created_utc"] = pd.to_datetime(df_amazon["timestamp"], unit="ms", errors="coerce")
    df_amazon = df_amazon.drop(columns=["timestamp"])
    
    print(f"Loaded {len(df_amazon)} Amazon reviews")
    print(f"Rating distribution:")
    print(df_amazon["rating"].value_counts().sort_index())
    df_amazon.head(3)

except Exception as e:
    print(f"Error loading Amazon dataset: {e}")
    print("Creating sample data for demonstration...")
    
    # Fallback: synthetic sample data for demonstration
    import random
    random.seed(42)
    sample_reviews = [
        (5, "Great product! The battery life is excellent and the screen is beautiful."),
        (5, "Amazing value for money. Fast shipping and works perfectly out of the box."),
        (4, "Good quality but the setup instructions could be clearer."),
        (3, "Decent product but nothing special. The build quality feels a bit cheap."),
        (2, "Disappointed with the battery life. It barely lasts 4 hours."),
        (1, "Stopped working after 2 weeks. Very poor quality. Do not buy."),
        (5, "Best purchase I've made this year! The camera quality is stunning."),
        (4, "Works well overall. The software has some bugs but the hardware is great."),
        (2, "The customer service was terrible when I had an issue."),
        (5, "Highly recommend! Fast, reliable, and great value."),
    ] * 50  # Repeat to get 500 reviews
    
    df_amazon = pd.DataFrame([
        {
            "review_id": str(i),
            "source": "amazon_sample",
            "rating": r,
            "full_text": t,
            "text": t,
            "helpful_vote": random.randint(0, 20),
            "verified_purchase": random.choice([True, False]),
            "created_utc": datetime.now() - timedelta(days=random.randint(0, 365)),
            "asin": f"B{random.randint(10000, 99999)}",
        }
        for i, (r, t) in enumerate(sample_reviews)
    ])
    
    print(f"Created {len(df_amazon)} sample reviews for demonstration")
    print(df_amazon["rating"].value_counts().sort_index())

In [ ]:
# ── 10.2 Review Sentiment Analysis ────────────────────────────────────────────

# Apply VADER
print("Analyzing review sentiment...")
review_sentiment = analyze_sentiment_vader(df_amazon["full_text"].tolist())
df_amazon = pd.concat([df_amazon.reset_index(drop=True), review_sentiment], axis=1)

# Compare star rating vs. NLP sentiment
df_amazon["rating_label"] = df_amazon["rating"].map({
    1: "negative", 2: "negative",
    3: "neutral",
    4: "positive", 5: "positive"
})

# Agreement rate
agreement = (df_amazon["rating_label"] == df_amazon["sentiment"]).mean()
print(f"\nVADER vs Star Rating agreement: {agreement:.1%}")
print("\nCross-tabulation (VADER sentiment × Star rating):")
print(pd.crosstab(df_amazon["sentiment"], df_amazon["rating"]))

In [ ]:
# Rating distribution plot
rating_counts = df_amazon["rating"].value_counts().sort_index().reset_index()
rating_counts.columns = ["Rating", "Count"]

fig = px.bar(
    rating_counts,
    x="Rating", y="Count",
    title="Amazon Review Rating Distribution",
    color="Rating",
    color_continuous_scale=["#e74c3c", "#e67e22", "#f39c12", "#2ecc71", "#27ae60"],
    template="plotly_white"
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# ── 10.3 Aspect-Based Analysis with spaCy ─────────────────────────────────────
# Extract noun-adjective pairs as a lightweight ABSA proxy

def extract_aspect_opinion_pairs(texts: list[str], n_sample: int = 200) -> pd.DataFrame:
    """
    Extract (aspect, opinion) pairs using dependency parsing.
    Noun = aspect candidate, its adjective modifiers = opinion candidates.
    """
    pairs = []
    texts_sample = texts[:n_sample]
    
    for doc in nlp.pipe(texts_sample, batch_size=50):
        for token in doc:
            # Find nouns that have adjective dependencies
            if token.pos_ in ["NOUN", "PROPN"]:
                adj_modifiers = [
                    child.text.lower()
                    for child in token.children
                    if child.pos_ == "ADJ"
                ]
                if adj_modifiers:
                    for adj in adj_modifiers:
                        pairs.append({
                            "aspect": token.lemma_.lower(),
                            "opinion": adj,
                            "context": token.sent.text[:100]
                        })
    
    return pd.DataFrame(pairs)


print("Extracting aspect-opinion pairs from reviews...")
df_aspects = extract_aspect_opinion_pairs(df_amazon["full_text"].tolist(), n_sample=300)

print(f"\nTotal aspect-opinion pairs: {len(df_aspects)}")
print("\nTop 15 most mentioned aspects:")
print(df_aspects["aspect"].value_counts().head(15))

In [ ]:
# Sentiment per aspect
def get_aspect_sentiment(aspect_df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """Get average VADER sentiment for the most common aspects."""
    sia = SentimentIntensityAnalyzer()
    
    aspect_df["opinion_sentiment"] = aspect_df["opinion"].apply(
        lambda x: sia.polarity_scores(str(x))["compound"]
    )
    
    top_aspects = aspect_df["aspect"].value_counts().head(top_n).index.tolist()
    
    result = (
        aspect_df[aspect_df["aspect"].isin(top_aspects)]
        .groupby("aspect")["opinion_sentiment"]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={"mean": "avg_sentiment", "count": "mention_count"})
        .sort_values("avg_sentiment", ascending=False)
    )
    
    return result


aspect_sentiment = get_aspect_sentiment(df_aspects)

# Color bars by sentiment
aspect_sentiment["color"] = aspect_sentiment["avg_sentiment"].apply(
    lambda x: "#2ecc71" if x > 0.05 else ("#e74c3c" if x < -0.05 else "#95a5a6")
)

fig = px.bar(
    aspect_sentiment,
    x="avg_sentiment", y="aspect",
    orientation="h",
    title="Aspect-Level Sentiment in Amazon Reviews",
    labels={"avg_sentiment": "Average Sentiment Score", "aspect": "Product Aspect"},
    color="avg_sentiment",
    color_continuous_scale=["#e74c3c", "#f39c12", "#2ecc71"],
    template="plotly_white",
    text="mention_count"
)
fig.update_traces(texttemplate="%{text} mentions", textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.show()

In [ ]:
# ── 10.4 Helpfulness Analysis ─────────────────────────────────────────────────

df_amazon["text_length"] = df_amazon["full_text"].str.len()

# Is longer review more helpful?
if "helpful_vote" in df_amazon.columns and df_amazon["helpful_vote"].max() > 0:
    fig = px.scatter(
        df_amazon.query("helpful_vote >= 0"),
        x="text_length",
        y="helpful_vote",
        color="rating",
        title="Review Length vs. Helpfulness Votes",
        labels={"text_length": "Review Length (chars)", "helpful_vote": "Helpful Votes"},
        color_continuous_scale="RdYlGn",
        template="plotly_white",
        opacity=0.5
    )
    fig.show()
    
    corr = df_amazon[["text_length", "helpful_vote"]].corr().iloc[0, 1]
    print(f"Correlation (text length vs. helpfulness): {corr:.3f}")

In [ ]:
# ── 10.5 Yelp Reviews (Optional — requires local dataset) ─────────────────────
# Download from: https://business.yelp.com/data/resources/open-dataset/
# Or load from HuggingFace:

print("Loading Yelp dataset sample...")
print("Note: Full Yelp dataset requires free registration at yelp.com/dataset")

try:
    # Load from HuggingFace
    yelp_dataset = load_dataset("Yelp/yelp_review_full", split="train", streaming=True)
    
    yelp_records = []
    for i, row in enumerate(yelp_dataset):
        if i >= 1000:
            break
        yelp_records.append({
            "source": "yelp",
            "rating": row["label"] + 1,  # 0-indexed → 1-5
            "full_text": row["text"],
            "text_length": len(row["text"]),
        })
    
    df_yelp = pd.DataFrame(yelp_records)
    print(f"Loaded {len(df_yelp)} Yelp reviews")
    print(df_yelp["rating"].value_counts().sort_index())

except Exception as e:
    print(f"Yelp dataset error: {e}")
    print("\nTo load locally from downloaded Yelp JSON:")
    print("""
    import json
    reviews = []
    with open('yelp_academic_dataset_review.json') as f:
        for line in f:
            reviews.append(json.loads(line))
    df_yelp = pd.DataFrame(reviews[:10000])  # Take first 10K for testing
    """)

---
## 11. Integrated Brand Health Dashboard

Combine all signals into a unified brand health view.

In [ ]:
# ── 11.1 Signal Summary ────────────────────────────────────────────────────────

def compute_brand_health_score(df_social: pd.DataFrame, df_reviews: pd.DataFrame) -> dict:
    """
    Compute a composite brand health score from social + review signals.
    Returns dict of metrics.
    """
    metrics = {}
    
    # --- Social Signals ---
    if not df_social.empty and "sentiment" in df_social.columns:
        total = len(df_social)
        pos = (df_social["sentiment"] == "positive").sum()
        neg = (df_social["sentiment"] == "negative").sum()
        
        metrics["social_post_count"] = total
        metrics["social_nss"] = round((pos - neg) / max(total, 1) * 100, 1)
        metrics["social_positive_pct"] = round(pos / max(total, 1) * 100, 1)
        metrics["social_negative_pct"] = round(neg / max(total, 1) * 100, 1)
        metrics["avg_virality_score"] = round(df_social["virality_score"].mean(), 2)
        metrics["top_post_score"] = int(df_social["score"].max())
    
    # --- Review Signals ---
    if not df_reviews.empty and "rating" in df_reviews.columns:
        metrics["review_count"] = len(df_reviews)
        metrics["avg_rating"] = round(df_reviews["rating"].mean(), 2)
        metrics["pct_5star"] = round((df_reviews["rating"] == 5).mean() * 100, 1)
        metrics["pct_1star"] = round((df_reviews["rating"] == 1).mean() * 100, 1)
        
        if "helpful_vote" in df_reviews.columns:
            metrics["avg_helpfulness"] = round(df_reviews["helpful_vote"].mean(), 2)
    
    # --- Composite Score (0-100) ---
    score_components = []
    
    if "social_nss" in metrics:
        # NSS: -100 to +100 → normalize to 0-100
        score_components.append((metrics["social_nss"] + 100) / 2 * 0.4)  # 40% weight
    
    if "avg_rating" in metrics:
        # Rating: 1-5 → normalize to 0-100
        score_components.append((metrics["avg_rating"] - 1) / 4 * 100 * 0.4)  # 40% weight
    
    if score_components:
        total_weight = 0.4 * len(score_components)
        metrics["brand_health_score"] = round(sum(score_components) / total_weight, 1)
    
    return metrics


health = compute_brand_health_score(df_social, df_amazon)

print(f"{'='*50}")
print(f"BRAND HEALTH REPORT — '{TOPIC_SHORT}'")
print(f"{'='*50}")
print(f"\nSOCIAL MEDIA SIGNALS")
print(f"  Total posts analyzed : {health.get('social_post_count', 'N/A')}")
print(f"  Net Sentiment Score  : {health.get('social_nss', 'N/A')} / 100")
print(f"  Positive posts       : {health.get('social_positive_pct', 'N/A')}%")
print(f"  Negative posts       : {health.get('social_negative_pct', 'N/A')}%")
print(f"  Avg virality score   : {health.get('avg_virality_score', 'N/A')}")
print(f"\nREVIEW SIGNALS")
print(f"  Total reviews        : {health.get('review_count', 'N/A')}")
print(f"  Average rating       : {health.get('avg_rating', 'N/A')} / 5.0")
print(f"  5-star reviews       : {health.get('pct_5star', 'N/A')}%")
print(f"  1-star reviews       : {health.get('pct_1star', 'N/A')}%")
print(f"\n{'─'*50}")
print(f"  COMPOSITE BRAND HEALTH SCORE: {health.get('brand_health_score', 'N/A')} / 100")
print(f"{'='*50}")

In [ ]:
# ── 11.2 Gauge Chart — Brand Health Score ─────────────────────────────────────

score = health.get("brand_health_score", 50)

fig = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=score,
    title={"text": f"Brand Health Score\n'{TOPIC_SHORT}'", "font": {"size": 18}},
    delta={"reference": 50, "increasing": {"color": "green"}, "decreasing": {"color": "red"}},
    gauge={
        "axis": {"range": [0, 100]},
        "bar": {"color": "#2c3e50"},
        "steps": [
            {"range": [0, 33], "color": "#e74c3c"},
            {"range": [33, 66], "color": "#f39c12"},
            {"range": [66, 100], "color": "#2ecc71"},
        ],
        "threshold": {
            "line": {"color": "red", "width": 4},
            "thickness": 0.75,
            "value": score
        }
    }
))
fig.update_layout(height=350, template="plotly_white")
fig.show()

In [ ]:
# ── 11.3 Social vs Review Signal Comparison ───────────────────────────────────

# Normalize signals to 0-100 scale for comparison
social_nss_norm = (health.get("social_nss", 0) + 100) / 2
review_rating_norm = (health.get("avg_rating", 3) - 1) / 4 * 100

categories = ["Social Sentiment", "Review Score", "Positive Social %", 
              "5-Star Review %", "Brand Health"]
values = [
    social_nss_norm,
    review_rating_norm,
    health.get("social_positive_pct", 0),
    health.get("pct_5star", 0),
    health.get("brand_health_score", 50)
]

fig = go.Figure(data=[
    go.Bar(
        x=categories,
        y=values,
        marker_color=["#3498db", "#2ecc71", "#3498db", "#2ecc71", "#9b59b6"],
        text=[f"{v:.1f}" for v in values],
        textposition="auto",
    )
])

fig.update_layout(
    title=f"Social + Review Signal Comparison (0-100 scale)",
    yaxis=dict(range=[0, 100]),
    template="plotly_white",
    showlegend=False
)

# Reference line at 50 (neutral)
fig.add_hline(y=50, line_dash="dash", line_color="gray",
              annotation_text="Neutral (50)")
fig.show()

In [ ]:
# ── 11.4 Share of Voice — Multi-Topic Comparison ──────────────────────────────
# Compare your topic against competitors using HackerNews (no auth needed)

def get_topic_mention_count(topic: str) -> int:
    """Get approximate mention count for a topic from HackerNews."""
    url = "https://hn.algolia.com/api/v1/search"
    params = {"query": topic, "tags": "story", "hitsPerPage": 1}
    try:
        r = requests.get(url, params=params, timeout=5)
        return r.json().get("nbHits", 0)
    except Exception:
        return 0


# Define topics to compare (change these to your competitors)
comparison_topics = [
    "OpenAI",
    "Anthropic",
    "Google Gemini",
    "Meta AI",
    "Microsoft Copilot"
]

print("Fetching mention counts for SOV analysis (HackerNews)...")
sov_data = {}
for topic in comparison_topics:
    count = get_topic_mention_count(topic)
    sov_data[topic] = count
    print(f"  {topic:25s}: {count:6,d} mentions")
    time.sleep(0.5)  # Be polite

total_mentions = sum(sov_data.values())
sov_df = pd.DataFrame([
    {"topic": k, "mentions": v, "sov_pct": v / max(total_mentions, 1) * 100}
    for k, v in sov_data.items()
]).sort_values("sov_pct", ascending=False)

fig = px.bar(
    sov_df,
    x="topic", y="sov_pct",
    title="Share of Voice (HackerNews mentions)",
    labels={"topic": "Brand/Topic", "sov_pct": "Share of Voice (%)"},
    color="sov_pct",
    color_continuous_scale="Blues",
    template="plotly_white",
    text="sov_pct"
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.show()

In [ ]:
# ── 11.5 Summary: What to Do Next ─────────────────────────────────────────────

print("""
========================================================
NEXT STEPS & EXTENSIONS
========================================================

DATA COLLECTION UPGRADES
  • Reddit PRAW: Add credentials to unlock full data collection
  • YouTube API: Add key for video comment analysis
  • Twitter/X: Consider $100/month Basic plan if X data is critical
  • GDELT: Enable multi-keyword queries for competitive analysis

NLP UPGRADES  
  • Swap VADER → Twitter-RoBERTa for higher accuracy
    model = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
  • Add ABSA with pyabsa for product-level feedback
  • Add multilingual support with XLM-T for global brands
  • Fine-tune BERT on domain-specific labeled examples

NETWORK ANALYSIS UPGRADES
  • Collect retweet graphs for cascade analysis
  • Add SEIR diffusion model (NDlib) with calibrated parameters
  • Network visualization with pyvis or gephi

REVIEW ANALYSIS UPGRADES
  • Download full Yelp Open Dataset (6.9M reviews)
  • Add fake review detection pipeline
  • Cross-correlate social sentiment leads with review score lags

PRODUCTIONIZATION
  • Schedule collection with cron / Airflow / Prefect
  • Store in PostgreSQL or Elasticsearch
  • Build real-time dashboard with Streamlit or Grafana
  • Add alert webhooks (Slack, email) for sentiment spikes

See 01_social_listening_overview.ipynb for full documentation.
""")